# Starlink Ku-Band Link Analysis
---

This notebook demonstrates a full satellite-to-ground link analysis using xant, modeling both ends of a Starlink Ku-band downlink as a worked example.

* Design the array lattice: derive the maximum grating-lobe-free element spacing for rectangular and triangular lattices at the Ku-band scan requirement, and visualize the visible region diagram for each
* Retrieve a live satellite pass: query a current Starlink TLE and build a time-varying coordinate system tracking the satellite over the Tetons
* Build the satellite array: construct a hierarchical triangular phased array representative of the Starlink V2 satellite aperture, with circularly polarized TE₁₀ aperture elements
* Build the ground terminal (Dishy): reconstruct the Starlink user terminal array geometry from publicly available teardown images, including the 19×2 subarray architecture and the 20° fixed tilt
* Steer both arrays: point the satellite array toward the ground terminal and Dishy toward the satellite, using time-dependent geodetic steering that tracks the satellite along its pass
* Visualize the patterns: project both steered array patterns onto an H3 hexagonal grid at the ground and at orbital altitude, and inspect the element phase progression across each aperture
* Compute received power: calculate the downlink received power over the satellite pass using free-space path loss

All coordinate transforms, antenna pointing, and polarization projections are handled automatically by xant and hics. The satellite coordinate system is live, rerunning the notebook will use the closest Starlink satellite at the current time.

# Phased Array Lattices
---


If the array is formed with a rectangular lattice, the maximum grating lobe free spacing is given by

$$
\begin{equation}
d_{max,rect.}=\frac{\lambda}{1+\sin\theta_{max}}
\end{equation}
$$

At 12.7 GHz and maximum scan of 60 degrees we get $d_{max,rectangular}=1.27$ cm.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

from datetime import UTC, datetime, timedelta, timezone

import numpy as np
import xarray as xr
from hics import HCS
from hics import plotting as hcsplt
from hics.geo import tle
from matplotlib import pylab as plt
from scipy.spatial.transform import Rotation

from xant import ureg
from xant.antenna import common, phasedarray
from xant.plotting import plot_antenna_pattern, plot_lm_h3
from hics import HICSLogger

# HICSLogger.unmute()
# HICSLogger.level = "DEBUG"
# from xant import XANTLogger
# XANTLogger.level = "DEBUG"
# XANTLogger.unmute()

In [ ]:
max_scan = 55 * ureg.degree
dxrect = 1 / (1 + np.sin(max_scan))
dxrect
ax = phasedarray.plot_grating_lobe_diagram(
    dxrect,
    dxrect,
    (0 * ureg.degree, 0 * ureg.degree),
    "rectangular",
    max_scan,
    figsize=None,
)

## Triangular lattice
---
However, we have seen from pictures of the ground terminal that a triangular lattice is used - and it would make sense that both the ground terminal and satellite would use the same lattice for the Ku-band aperture. There are advantages of triangular lattices because each cell is a hexagon which efficiently tiles the aperture face using fewer elements (approximately 15% fewer) for the same maximum scanning. The maximum grating lobe free spacing for a triangular array is given by

$$
\begin{equation}
d_{max,tri.}=\frac{2}{\sqrt{3}}\left(\frac{\lambda}{1+\sin\theta_{max}}\right)
\end{equation}
$$

Generally, the lattice space is slightly smaller than the maximum spacing as the edge of the grating lobe will start to appear reducing the gain. So for this example let's assume the spacing is
$d_{triangular}=0.61\lambda =1.44$ cm.

In [ ]:
dxtriang = dxrect * 2 / np.sqrt(3)
ax = phasedarray.plot_grating_lobe_diagram(
    dxtriang,
    dxtriang,
    (0 * ureg.degree, 0 * ureg.degree),
    "triangular",
    max_scan,
    figsize=None,
)

# Setup Coordinate System

Create a coordinate system from a Starlink Satelitte based on TLE data.
Get satellite closest to Bould right now.
Create a 

In [ ]:
boulder_latlon = (40.0150, -105.2705)
tetons_latlon = (43.912241, -110.642212)
dishy_base = HCS.from_crs(
    (tetons_latlon[0] * ureg.degree, tetons_latlon[1] * ureg.degree, 10 * ureg.ft),
    hagl=True,
)
time_search = datetime.now().astimezone().astimezone(UTC)
closest_satellite, sats = tle.get_closest_from_tle(
    *tetons_latlon, "starlink", time_search, update_tle=False
)
starlinkcs = tle.satellite_to_cs(
    closest_satellite,
    t0=time_search,
    npts=11,
    tdelta=timedelta(seconds=10),
)
m = hcsplt.showcs_leafmap(starlinkcs)
m = hcsplt.showcs_leafmap(dishy_base, m=m)
# m.to_html("tmp_map.html", title="Starlink CS")
# import webbrowser, pathlib

# webbrowser.open(pathlib.Path("tmp_map.html").resolve().as_uri())

## Setup RF Parameters and Array

In [ ]:
f0 = 11.7 * ureg.GHz
lam0 = (ureg.speed_of_light / f0).to("cm")
lam_min = (ureg.speed_of_light / (12.7 * ureg.GHz)).to("cm")
max_scan = 60 * ureg.degree
l = lam_min / 2
# dx =dxtriang*lam_min # This is approximately 1.5cm
dx = 1.5 * ureg.cm
f0 = [11.7] * ureg.GHz

# Element Efficiency
el_eff = np.sqrt(0.8)  # Assume 80% efficient

## Create Satellite Antenna

In [ ]:
csref = HCS((0, 0, 0) * ureg.m, reference=starlinkcs)
csref90 = HCS(
    (0, 0, 0) * ureg.m,
    rotation=Rotation.from_euler("ZYZ", [90, 0, 0], degrees=True),
    reference=csref,
)
# H/V Constituent Elements
e0 = common.TE10Aperture(l, l, f0, hcs=csref)
e90 = common.TE10Aperture(l, l, f0, hcs=csref90)
# RH and LH elements
erh = (e0 - e90 * 1j) * el_eff / np.sqrt(2)
elh = (e0 + e90 * 1j) * el_eff / np.sqrt(2)

nx = 32
ny = 38

nsubx = 16
nsuby = 2

sub_arr = phasedarray.AntennaArray.triangular(erh, nsubx, nsuby, dx, cs_reference=csref)

nfullx = int(nx / nsubx)
nfully = int(ny / nsuby)

fulldx = nsubx * dx
fulldy = nsuby * dx * np.sqrt(3) / 2

full = phasedarray.AntennaArray.rectangular(
    sub_arr,
    nfullx,
    fulldx,
    nfully,
    fulldy,
    cs_reference=csref,
)

ax = full.showxy(
    alpha=1,
    rs=1.22,
    unit_shape="hexagon",
)
bound = plt.Rectangle((-25, -25), 50, 50, fill=False)
ax.add_patch(bound)
ax.grid(False)
ax.set_title("Starlink Satellite Terminal\nKu-Band Array Element Positions")
plt.tight_layout()

In [ ]:
idx = 3
req = dict(theta=np.linspace(-90, 90, 1801) * ureg.degree, phi=0 * ureg.degree)
a = erh.request_data(**req)
ax = plot_antenna_pattern(a.squeeze().isel(time=idx).sel(polarization="rhcp"), x="theta")

## Create Dishy Ground Antenna

In [ ]:
dishy_face = HCS(
    (0, 0, 0) * ureg.m,
    reference=dishy_base,
    rotation=Rotation.from_euler("ZYZ", [90, -20, 0], degrees=True),
)  # Looks like it says to pivot 20deg off zenith

csref_gnd = HCS((0, 0, 0) * ureg.m, reference=dishy_face)
csref90_gnd = HCS(
    (0, 0, 0) * ureg.m,
    rotation=Rotation.from_euler("ZYZ", [90, 0, 0], degrees=True),
    reference=csref_gnd,
)
# H/V Constituent Elements
e0g = common.TE10Aperture(l, l, f0, hcs=csref_gnd)
e90g = common.TE10Aperture(l, l, f0, hcs=csref90_gnd)
# Circularly Polarized Element
egrh = (e0g - e90g * 1j) * el_eff / np.sqrt(2)

# https://www.youtube.com/watch?v=AlvIWF0AXI0&t=781s
# 38 x 27 (approx) call it 38x28
nx_dishy = 38
ny_dishy = 28
nsubx_dishy = 19
nsuby_dishy = 2

sub_arr_dishy = phasedarray.AntennaArray.triangular(
    egrh,
    nsubx_dishy,
    nsuby_dishy,
    dx,
    cs_reference=csref_gnd,
)

nfullx_dishy = int(nx_dishy / nsubx_dishy)
nfully_dishy = int(ny_dishy / nsuby_dishy)

fulldx_dishy = nsubx_dishy * dx
fulldy_dishy = nsuby_dishy * dx * np.sqrt(3) / 2

full_dishy = phasedarray.AntennaArray.rectangular(
    sub_arr_dishy,
    nfullx_dishy,
    fulldx_dishy,
    nfully_dishy,
    fulldy_dishy,
    cs_reference=csref_gnd,
)

ax = full_dishy.showxy(
    alpha=1,
    rs=1.22,
    unit_shape="hexagon",
)
bound = plt.Rectangle((-59.4 / 2, -38.3 / 2), 59.4, 38.2, fill=False)
ax.add_patch(
    bound,
)
ax.grid(False)
ax.set_title("Starlink Ground Terminal Ku-Band Array Element Positions")
plt.tight_layout()

## Local Array CS View

In [ ]:
pltter = hcsplt.viewcs(full_dishy.coordinate_systems[0].reference, dishy_base, vector_length=50000)
pltter = hcsplt.viewcs(full.coordinate_systems[0], dishy_base, ax=pltter, vector_length=50000)
pltter.show()

## Plot Broadside Patterns

In [ ]:
phasedarray.steer_phase_centers(
    full_dishy, f0[0], coordinate_frame="phitheta", phi=0 * ureg.degree, theta=0 * ureg.degree
)
idx = 3
req = dict(theta=np.linspace(-90, 90, 1801) * ureg.degree, phi=0 * ureg.degree)
a = full_dishy.total.request_data(**req)
ax = plot_antenna_pattern(a.squeeze().sel(polarization="rhcp"), x="theta")


phasedarray.steer_phase_centers(
    full, f0[0], coordinate_frame="phitheta", phi=0 * ureg.degree, theta=0 * ureg.degree
)
idx = 3
req = dict(theta=np.linspace(-90, 90, 1801) * ureg.degree, phi=0 * ureg.degree)
a = full.total.request_data(**req)
ax = plot_antenna_pattern(a.squeeze().isel(time=idx).sel(polarization="rhcp"), x="theta")

## Steer satellite towards ground terminal

In [ ]:
lat = xr.zeros_like(starlinkcs.origin.time)
lat.data = ([dishy_base.llh[0].item().magnitude] * lat.size) * ureg.degree
lon = xr.zeros_like(starlinkcs.origin.time)
lon.data = ([dishy_base.llh[1].item().magnitude] * lon.size) * ureg.degree
h = xr.zeros_like(starlinkcs.origin.time)
h.data = ([dishy_base.llh[2].item().magnitude] * h.size) * ureg.m

sp0, st0 = phasedarray.steer_phase_centers(
    full,
    f0[0],
    coordinate_frame="llh",
    lat=lat,
    lon=lon,
    h=h,
    convert_kwargs=dict(hagl=False),
)

## Steer ground terminal towards satellite

In [ ]:
stl = starlinkcs.llh
gp0, gt0 = phasedarray.steer_phase_centers(
    full_dishy,
    f0[0],
    coordinate_frame="llh",
    lat=stl[0],
    lon=stl[1],
    h=stl[2],
    convert_kwargs=dict(hagl=False),
)

## Project pattern on the ground

Resolution 5 H3 cell

In [ ]:
# Satellite
dh3 = full.total.request_data(
    i=(np.arange(81) - 40) * ureg.dimensionless,
    j=(np.arange(81) - 40) * ureg.dimensionless,
    h=6800 * ureg.ft,
    coordinate_frame="h3",
    convert_kwargs=dict(origin_latlong=tetons_latlon, resolution=6, hagl=False),
)
dh3 = dh3.sel(polarization="rhcp").squeeze()
idx = 3
scs = starlinkcs.llh
satcsidx = HCS.from_crs((scs[0].isel(time=idx), scs[1].isel(time=idx), scs[2].isel(time=idx)))
m = plot_lm_h3(dh3.isel(time=idx), tetons_latlon)
hcsplt.showcs_leafmap(dishy_base, m=m)
hcsplt.showcs_leafmap(satcsidx, m=m)
m

## Project Dishy to LEO orbit to see pattern

In [ ]:
# Dishy
dh3 = full_dishy.total.request_data(
    i=(np.arange(81) - 40) * ureg.dimensionless,
    j=(np.arange(81) - 40) * ureg.dimensionless,
    h=starlinkcs.llh[-1].mean().item(),
    coordinate_frame="h3",
    convert_kwargs=dict(origin_latlong=tetons_latlon, resolution=6, hagl=False),
)
dh3 = dh3.sel(polarization="rhcp").squeeze()
idx = 3
scs = starlinkcs.llh
satcsidx = HCS.from_crs((scs[0].isel(time=idx), scs[1].isel(time=idx), scs[2].isel(time=idx)))
m = plot_lm_h3(dh3.isel(time=idx), tetons_latlon)
hcsplt.showcs_leafmap(dishy_base, m=m)
hcsplt.showcs_leafmap(satcsidx, m=m)
m

## Dishy Pattern Cut

In [ ]:
idx = 3
req = dict(theta=np.linspace(-90, 90, 1801) * ureg.degree, phi=gp0.isel(time=idx).item())
a = full_dishy.total.request_data(**req)
ax = plot_antenna_pattern(a.squeeze().isel(time=idx).sel(polarization="rhcp"), x="theta")

## View Resultant Element Phase

In [ ]:
ax = full.showxy(
    alpha=1,
    rs=1.22,
    unit_shape="hexagon",
    color_type="phase",
    exc_isel=dict(time=idx),
)
bound = plt.Rectangle((-25, -25), 50, 50, fill=False)
ax.add_patch(bound)
ax.grid(False)
ax.set_title("Starlink Satellite Terminal\nKu-Band Array Phase Progression")
plt.tight_layout()

In [ ]:
ax = full_dishy.showxy(
    alpha=1,
    rs=1.22,
    unit_shape="hexagon",
    color_type="phase",
    exc_isel=dict(time=idx),
)
bound = plt.Rectangle((-59.4 / 2, -38.3 / 2), 59.4, 38.2, fill=False)
ax.add_patch(
    bound,
)
ax.grid(False)
ax.set_title("Starlink Ground Terminal Ku-Band Array Phase Progression")
plt.tight_layout()

## Calculate Received Power

In [ ]:
from xant.propagation import rflink

In [ ]:
tx_power = 28.7 * ureg.dBm
res, txcs, rxcs = rflink.calculate_spatial_link(
    full.total,
    tx_power,
    full_dishy.total,
    propagation="fspl",
)

In [ ]:
fig, ax = plt.subplots()
res.rx_power.plot(ax=ax)
ax.set_ylim(-85, -65)
ax.set_title("Downlink RX Power")